In [ ]:
import json
import os
import sys
from litellm import completion
from typing import List, Dict

def extract_markdown_block(response: str, block_type: str = "json") -> str:
    """Extract code block from response"""

    if not '```' in response:
        return response

    code_block = response.split('```')[1].strip()

    if code_block.startswith(block_type):
        code_block = code_block[len(block_type):].strip()

    return code_block

# DE 1024 A 2048 porque como ahora no es json si no en texto consume mas tokens
import time

def generate_response(messages: List[Dict], retries: int = 3, backoff: float = 3.0) -> str:
    """Call LLM to get a response."""
    for intento in range(retries):
        try:
            response = completion(
                model="groq/openai/gpt-oss-120b",
                messages=messages,
                max_tokens=1024,
                reasoning_effort="low"
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            print(f"Intento {intento+1} falló: {type(e).__name__}: {e}")
            if intento < retries - 1:
                time.sleep(backoff)
            else:
                raise

def parse_action(response: str) -> Dict:
    """Parse the LLM response into a structured action dictionary."""
    try:
        response = extract_markdown_block(response, "action")
        response_json = json.loads(response)
        if "tool_name" in response_json and "args" in response_json:
            return response_json
        else:
            return {"tool_name": "error", "args": {"message": "You must respond with a JSON tool invocation."}}
    except json.JSONDecodeError:
        return {"tool_name": "error", "args": {"message": "Invalid JSON response. You must respond with a JSON tool invocation."}}

def list_files() -> List[str]:
    """List files in the current directory."""
    return os.listdir(".")

def read_file(file_name: str, max_chars: int = 3000) -> str:
    """Read a file's contents."""
    try:
        with open(file_name, "r") as file:
            content = file.read()
        if len(content) > max_chars:
            content = content[:max_chars] + f"\n\n... [truncado, archivo completo tiene {len(content)} caracteres]"
        return content
    except FileNotFoundError:
        return f"Error: {file_name} not found."
    except Exception as e:
        return f"Error: {str(e)}"

# Define system instructions (Agent Rules)
agent_rules = [{
    "role": "system",
    "content": """
You are an AI agent that can perform tasks by using available tools.

Available tools:

```json
{
    "list_files": {
        "description": "Lists all files in the current directory.",
        "parameters": {}
    },
    "read_file": {
        "description": "Reads the content of a file.",
        "parameters": {
            "file_name": {
                "type": "string",
                "description": "The name of the file to read."
            }
        }
    },
    "terminate": {
        "description": "Ends the agent loop and provides a summary of the task.",
        "parameters": {
            "message": {
                "type": "string",
                "description": "Summary message to return to the user."
            }
        }
    }
}
```

If a user asks about files, documents, or content, first list the files before reading them.

When you are done, terminate the conversation by using the "terminate" tool and I will provide the results to the user.

Important!!! Every response MUST have an action.
You must ALWAYS respond in this format:

<Stop and think step by step. Parameters map to args. Insert a rich description of your step by step thoughts here.>

```action
{
    "tool_name": "insert tool_name",
    "args": {...fill in any required arguments here...}
}
```"""
}]

# Initialize agent parameters
iterations = 0
max_iterations = 10

user_task = input("What would you like me to do? ")

memory = [{"role": "user", "content": user_task}]

# The Agent Loop
while iterations < max_iterations:
    # 1. Construct prompt: Combine agent rules with memory
    prompt = agent_rules + memory

    # 2. Generate response from LLM
    print("Agent thinking...")
    response = generate_response(prompt)
    print(f"Agent response: {response}")

    # 3. Parse response to determine action
    action = parse_action(response)
    result = "Action executed"

    if action["tool_name"] == "list_files":
        result = {"result": list_files()}
    elif action["tool_name"] == "read_file":
        result = {"result": read_file(action["args"]["file_name"])}
    elif action["tool_name"] == "error":
        result = {"error": action["args"]["message"]}
    elif action["tool_name"] == "terminate":
        print(action["args"]["message"])
        break
    else:
        result = {"error": "Unknown action: " + action["tool_name"]}

    print(f"Action result: {result}")

    # 5. Update memory with response and results
    memory.extend([
        {"role": "assistant", "content": response},
        {"role": "user", "content": json.dumps(result)}
    ])

    # 6. Check termination condition
    if action["tool_name"] == "terminate":
        break

    iterations += 1


Agent thinking...
Agent response: Explicaré el contenido del cuaderno 01_introduction.ipynb una vez lo haya leído. 

```action
{
    "tool_name": "list_files",
    "args": {}
}
```
Action result: {'result': ['01_introduction.ipynb', 'funcionmoduloi.py', '02_buildingAgent.ipynb', '03_agentWithTools.ipynb']}
Agent thinking...
Agent response: Ahora leeré el archivo 01_introduction.ipynb para describir su contenido.

```action
{
    "tool_name": "read_file",
    "args": {
        "file_name": "01_introduction.ipynb"
    }
}
```
Action result: {'result': '{\n "cells": [\n  {\n   "cell_type": "code",\n   "execution_count": null,\n   "id": "b6dae6ce",\n   "metadata": {},\n   "outputs": [\n    {\n     "name": "stdout",\n     "output_type": "stream",\n     "text": [\n      "Key cargada: True\\n"\n     ]\n    }\n   ],\n   "source": [\n    "from dotenv import load_dotenv, find_dotenv\\n",\n    "load_dotenv(find_dotenv())\\n",\n    "\\n",\n    "import os\\n",\n    "# Verificación rápida (opcional)

In [6]:
# ENFOQUE NATIVO
# En la version anterior se declara las herramientas con un simple prompt / esta usando el parametro tools el cual las estructura
# tambien el modelo responde con ```action / en el otro caso existe un campo tool_calls estructurando la respuesta
# como se extrae la accion Regex + json.loads manual / mientras que aqui ya viene parseado (tool.function.name/arguments)
# la ejecucion del tool es con puro if-elif / utiliza un diccionario tool_functions[nombre](**args)
import json
import os
from typing import List

from litellm import completion

def list_files() -> List[str]:
    """List files in the current directory."""
    return os.listdir(".")

def read_file(file_name: str, max_chars: int = 3000) -> str:
    """Read a file's contents."""
    try:
        with open(file_name, "r") as file:
            content = file.read()
        if len(content) > max_chars:
            content = content[:max_chars] + f"\n\n... [truncado, archivo completo tiene {len(content)} caracteres]"
        return content
    except FileNotFoundError:
        return f"Error: {file_name} not found."
    except Exception as e:
        return f"Error: {str(e)}"

def terminate(message: str) -> None:
    """Terminate the agent loop and provide a summary message."""
    print(f"Termination message: {message}")

tool_functions = {
    "list_files": list_files,
    "read_file": read_file,
    "terminate": terminate
}

tools = [
    {
        "type": "function",
        "function": {
            "name": "list_files",
            "description": "Returns a list of files in the directory.",
            "parameters": {"type": "object", "properties": {}, "required": []}
        }
    },
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "Reads the content of a specified file in the directory.",
            "parameters": {
                "type": "object",
                "properties": {"file_name": {"type": "string"}},
                "required": ["file_name"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "terminate",
            "description": "Terminates the conversation. No further actions or interactions are possible after this. Prints the provided message for the user.",
            "parameters": {
                "type": "object",
                "properties": {
                    "message": {"type": "string"},
                },
                "required": ["message"]
            }
        }
    }
]

agent_rules = [{
    "role": "system",
    "content": """
You are an AI agent that can perform tasks by using available tools. 

If a user asks about files, documents, or content, first list the files before reading them.

When you are done, terminate the conversation by using the "terminate" tool and I will provide the results to the user.
"""
}]

# Initialize agent parameters
iterations = 0
max_iterations = 10

user_task = input("What would you like me to do? ")

memory = [{"role": "user", "content": user_task}]

# The Agent Loop
while iterations < max_iterations:

    messages = agent_rules + memory

    response = completion(
        model="groq/openai/gpt-oss-120b",
        messages=messages,
        tools=tools,
        max_tokens=1024,
        reasoning_effort="low"
    )

    if response.choices[0].message.tool_calls:
        tool = response.choices[0].message.tool_calls[0]
        tool_name = tool.function.name
        tool_args = json.loads(tool.function.arguments)

        action = {
            "tool_name": tool_name,
            "args": tool_args
        }

        if tool_name == "terminate":
            print(f"Termination message: {tool_args['message']}")
            break
        elif tool_name in tool_functions:
            try:
                result = {"result": tool_functions[tool_name](**tool_args)}
            except Exception as e:
                result = {"error":f"Error executing {tool_name}: {str(e)}"}
        else:
            result = {"error": f"Unknown tool: {tool_name}"}

        print(f"Executing: {tool_name} with args {tool_args}")
        print(f"Result: {result}")
        memory.extend([
            {"role": "assistant", "content": json.dumps(action)},
            {"role": "user", "content": json.dumps(result)}
        ])
    else:
        result = response.choices[0].message.content
        print(f"Response: {result}")
        break

Executing: list_files with args {}
Result: {'result': ['01_introduction.ipynb', 'funcionmoduloi.py', '02_buildingAgent.ipynb', '03_agentWithTools.ipynb']}
Executing: read_file with args {'file_name': '01_introduction.ipynb'}
Result: {'result': '{\n "cells": [\n  {\n   "cell_type": "code",\n   "execution_count": null,\n   "id": "b6dae6ce",\n   "metadata": {},\n   "outputs": [\n    {\n     "name": "stdout",\n     "output_type": "stream",\n     "text": [\n      "Key cargada: True\\n"\n     ]\n    }\n   ],\n   "source": [\n    "from dotenv import load_dotenv, find_dotenv\\n",\n    "load_dotenv(find_dotenv())\\n",\n    "\\n",\n    "import os\\n",\n    "# Verificación rápida (opcional) - no imprimas la key completa nunca\\n",\n    "print(\\"Key cargada:\\", os.environ.get(\'OPENAI_API_KEY\') is not None)"\n   ]\n  },\n  {\n   "cell_type": "code",\n   "execution_count": 15,\n   "id": "562d22aa",\n   "metadata": {},\n   "outputs": [\n    {\n     "name": "stdout",\n     "output_type": "stream",